# 04b - Feature Engineering (3-year panel, with trend features)

**Goal:** extend the feature set using `data/processed/ward_panel_3yr.csv`, which adds 2011 as a third historical point. This unlocks *trend* features (is a ward's turnout rising or falling, is registration growing) that a two-election panel can't support without leaking the label — see the note below for exactly why.

**This notebook does not replace `04_feature_engineering.ipynb`** — that pipeline (on the full 4,344-ward `ward_panel.csv`) is still valid and complete on its own. This is an additional, richer feature set on a smaller ward set (3,834 wards, since matching now has to survive two demarcation cycles instead of one). Person C can decide which table to actually train on, or try both.

## Why trend features need a third timepoint

The rule for a safe feature isn't "no deltas allowed" — it's **"a feature can't be built from the same period as the label it's predicting."**

- `Turnout_2021 - Turnout_2016` predicting `Turnout_2021` = leakage (the label is baked into the feature)
- `Turnout_2016 - Turnout_2011` predicting `Turnout_2021` = **safe** — both 2016 and 2011 are fully known before 2021 happens

And the same shape carries cleanly to deployment: `Turnout_2021 - Turnout_2016` predicting the unknown `Turnout_2026` is equally safe, since 2021 and 2016 are both already-completed elections by the time we forecast 2026.

**Input:** `data/processed/ward_panel_3yr.csv` (3,834 wards, with `Turnout_2011/2016/2021`, `RegisteredVoters_2011/2016/2021`, and precomputed `TurnoutDelta_2011_2016` / `RegistrationGrowth_2011_2016`).

**Output:**
- `data/processed/train_features_3yr.csv` — 2016 snapshot + 2011→2016 trend, target = known Turnout_2021
- `data/processed/forecast_features_3yr.csv` — 2021 snapshot + 2016→2021 trend (computed here, since nobody had precomputed *this* lookback pair), no target (Turnout_2026 unknown)


In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.features import build_snapshot_features, build_trend_features

pd.set_option("display.max_columns", None)

panel3 = pd.read_csv("../data/processed/ward_panel_3yr.csv")
panel3.shape

(3834, 14)

## A quick reality check before building anything

Worth knowing up front: `TurnoutDelta_2011_2016` and `RegistrationGrowth_2011_2016` are both *individually* weak predictors of Turnout_2021 (r = 0.05 and 0.04 respectively, versus r = 0.68 for Turnout_2016 alone). They're leakage-safe and worth including, but don't expect them to transform the model on their own — this is an honest limitation worth stating plainly rather than overselling.

In [2]:
panel3[["Turnout_2011", "Turnout_2016", "TurnoutDelta_2011_2016", "RegistrationGrowth_2011_2016"]].corrwith(panel3["Turnout_2021"])

Turnout_2011                    0.624129
Turnout_2016                    0.682380
TurnoutDelta_2011_2016          0.050539
RegistrationGrowth_2011_2016    0.041407
dtype: float64

## Step 1 - Build the training table (2016 snapshot + 2011->2016 trend -> known 2021 outcome)

In [3]:
# No 2011 raw station-level files are available, so SpoiltRatio can't be
# recovered for this year - pass an empty lookup and drop the resulting
# all-missing column rather than leave broken NaNs in the feature table.
empty_spoilt = pd.DataFrame({"Province": [], "Ward": [], "SpoiltRatio": []})

snapshot_2016 = build_snapshot_features(panel3, empty_spoilt, year=2016).drop(columns=["SpoiltRatio_prior"])
train_features_3yr = build_trend_features(panel3, snapshot_2016, year=2016, prior_year=2011)

train_features_3yr = train_features_3yr.merge(
    panel3[["Province", "Ward", "Turnout_2021"]], on=["Province", "Ward"], how="left"
).rename(columns={"Turnout_2021": "Target_Turnout"})

print(train_features_3yr.shape)
train_features_3yr.head()

(3834, 12)


,Province,Ward,MunicipalityCode,RegisteredVoters_prior,Turnout_prior,IsMetro,LogRegisteredVoters_prior,MunicipalityAvgTurnout_prior,SnapshotYear,TurnoutDelta_prior,RegistrationGrowth_prior,Target_Turnout
0,Eastern Cape,Ward 29200001,BUF,8851,0.570896,True,9.088286,0.555891,2016,0.019465,0.319469,0.400459
1,Eastern Cape,Ward 29200002,BUF,7794,0.466513,True,8.961109,0.558022,2016,-0.055967,0.134003,0.441672
2,Eastern Cape,Ward 29200003,BUF,8118,0.416975,True,9.001839,0.559033,2016,0.032786,0.125936,0.338520
3,Eastern Cape,Ward 29200004,BUF,9175,0.674223,True,9.124238,0.553783,2016,-0.001118,0.182041,0.491476
4,Eastern Cape,Ward 29200005,BUF,9228,0.560360,True,9.129998,0.556106,2016,-0.012151,0.145766,0.403802


In [4]:
assert train_features_3yr.isna().sum().sum() == 0, "Unexpected missing values"
train_features_3yr.drop(columns=["Province", "Ward", "MunicipalityCode", "SnapshotYear"]).corrwith(train_features_3yr["Target_Turnout"])

RegisteredVoters_prior         -0.196619
Turnout_prior                   0.682380
IsMetro                        -0.172769
LogRegisteredVoters_prior      -0.245394
MunicipalityAvgTurnout_prior    0.402978
TurnoutDelta_prior              0.050539
RegistrationGrowth_prior        0.041407
Target_Turnout                  1.000000
dtype: float64

## Step 2 - Build the forecast table (2021 snapshot + 2016->2021 trend, no target yet)

In [5]:
snapshot_2021 = build_snapshot_features(panel3, empty_spoilt, year=2021).drop(columns=["SpoiltRatio_prior"])
forecast_features_3yr = build_trend_features(panel3, snapshot_2021, year=2021, prior_year=2016)

print(forecast_features_3yr.shape)
forecast_features_3yr.head()

(3834, 11)


,Province,Ward,MunicipalityCode,RegisteredVoters_prior,Turnout_prior,IsMetro,LogRegisteredVoters_prior,MunicipalityAvgTurnout_prior,SnapshotYear,TurnoutDelta_prior,RegistrationGrowth_prior
0,Eastern Cape,Ward 29200001,BUF,9589,0.400459,True,9.168372,0.451457,2021,-0.170437,0.083380
1,Eastern Cape,Ward 29200002,BUF,7655,0.441672,True,8.943114,0.450616,2021,-0.024841,-0.017834
2,Eastern Cape,Ward 29200003,BUF,9961,0.338520,True,9.206433,0.452721,2021,-0.078454,0.227026
3,Eastern Cape,Ward 29200004,BUF,9327,0.491476,True,9.140669,0.449599,2021,-0.182747,0.016567
4,Eastern Cape,Ward 29200005,BUF,8732,0.403802,True,9.074750,0.451388,2021,-0.156558,-0.053749


In [6]:
assert forecast_features_3yr.isna().sum().sum() == 0, "Unexpected missing values"
print("No missing values - OK to save.")

No missing values - OK to save.


## Step 3 - Save both tables

In [ ]:
train_features_3yr.to_csv("../data/processed/train_features_3yr.csv", index=False)
forecast_features_3yr.to_csv("../data/processed/forecast_features_3yr.csv", index=False)

print(f"Saved {len(train_features_3yr)} rows to data/processed/train_features_3yr.csv")
print(f"Saved {len(forecast_features_3yr)} rows to data/processed/forecast_features_3yr.csv")

Saved 3834 rows to data/processed/train_features_3yr.csv
Saved 3834 rows to data/processed/forecast_features_3yr.csv


## Handoff notes

**Two feature sets now exist** — Person C's call on which to use (or try both and compare):

| | `train_features.csv` (04) | `train_features_3yr.csv` (04b) |
|---|---|---|
| Wards | 4,344 | 3,834 (~12% fewer — more demarcation-cycle survival required) |
| Features | Turnout, RegisteredVoters, IsMetro, MunicipalityAvgTurnout, SpoiltRatio, Province | Same, minus SpoiltRatio (no 2011 raw data), plus TurnoutDelta_prior, RegistrationGrowth_prior |
| Trend info | None | Yes, but individually weak (r ≈ 0.04–0.05) |

**Known limitations to carry into the write-up:**
- `SpoiltRatio_prior` isn't available on the 3-year panel — would need the raw 2011 station-level files, which weren't provided.
- Trend features are leakage-safe but weak individually; worth checking in modelling whether they add anything once combined with the level features, and reporting honestly either way.
- Fewer wards here means slightly less coverage than the full panel — worth flagging if the two feature sets give meaningfully different results.